# Train Transolver on the SU2 hypersonic dataset

Kaggle orchestrator. Attach the `zeteixeira/su2-hypersonic-sphere-cone`
dataset as input (must be the 727-case version: sweep complete, all four
OOD slabs populated), set Accelerator to GPU T4 x2, Internet on, Run All.

Trains the **v2 baseline ensemble**: five Transolvers at `M = 32`, same
split seed, differing only in `--init-seed` (0-4), all trained from scratch
on the full 727-case dataset. This supersedes the W2 ensemble (which was
trained on the earlier 555-case version) and exists specifically as the
"before" side of the W4 active-learning loop control: v3 will be this same
recipe plus the loop-acquired cases, so the only difference between v2 and
v3 is the loop data, not an unrelated dataset-size confound. v2 also becomes
the published/dashboard model.

Unlike the earlier W2 runs, member 0 (`init_seed=0`) is **not** reused from
any prior checkpoint -- everything here is a fresh run on the new dataset.
Outputs land in `/kaggle/working/run_m32_v2_s{s}/` and survive the session
as notebook output.

Per-epoch cost is higher than the 555-case runs (train split grew from 384
to 481 core cases, roughly +25%), so budget conservatively: two members per
9h session is not guaranteed to fit anymore. Split `INIT_SEEDS` across
sessions as `[0, 1]`, `[2, 3]`, `[4]`, checking the first run's
`wallclock_s` in `history.json` before assuming a second member fits in the
same session.</cell_id>
<parameter name="cell_type">markdown

In [ ]:
import os, subprocess, sys, zipfile

REPO = "/kaggle/working/transolver-hypersonic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/zeteixeira03/transolver-hypersonic.git", REPO],
                   check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops"], check=True)

# find the dataset anywhere under /kaggle/input: kaggle has used both
# /kaggle/input/<slug>/ and /kaggle/input/datasets/<owner>/<slug>/ layouts,
# so walk the tree and score directories by their contents
INPUT = "/kaggle/input"
assert os.path.isdir(INPUT), "no /kaggle/input at all: attach the dataset"
DATA, best = None, -1
for root, dirs, files in os.walk(INPUT, followlinks=True):
    marker = 1000 if ("ledger.db" in files or "su2_cases.zip" in files) else 0
    score = marker + sum(1 for f in files if f.startswith("case_"))
    if score > best:
        best, DATA = score, root
print("selected:", DATA, "score:", best)
assert DATA is not None and best >= 1000, (
    f"no directory with ledger.db or su2_cases.zip under {INPUT}; "
    f"attach the su2 dataset (right panel -> Input -> Add Input)"
)

# the dataset version carries ledger.db at the root plus su2_cases.zip; kaggle
# may or may not auto-extract the zip. extract iff the case files themselves
# are not already present (ledger.db alone is not enough to train on)
n_root_cases = sum(1 for f in os.listdir(DATA) if f.startswith("case_"))
if n_root_cases == 0:
    zsrc = f"{DATA}/su2_cases.zip"
    assert os.path.isfile(zsrc), f"no case_* files and no su2_cases.zip in {DATA}"
    extracted = "/kaggle/working/su2_data"
    if not os.path.isfile(f"{extracted}/ledger.db"):
        os.makedirs(extracted, exist_ok=True)
        with zipfile.ZipFile(zsrc) as z:
            z.extractall(extracted)
        # the zip holds only case_*.npz; the ledger rides alongside at DATA root
        import shutil
        shutil.copyfile(f"{DATA}/ledger.db", f"{extracted}/ledger.db")
    DATA = extracted

n_cases = len([d for d in os.listdir(DATA) if d.startswith("case_")])
print("cases:", n_cases)
assert n_cases > 700, f"only {n_cases} case files under {DATA}; wrong dataset version attached? (expect 727)"

In [ ]:
import subprocess

INIT_SEEDS = [0, 1]         # session A; set to [2, 3] for session B, [4] for session C
M = 32                      # ensemble slice count (in-distribution optimum from the ablation)
EPOCHS = 350

def run(cmd):
    # stream child output into the cell; ! magics can't loop with error checks
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"run failed: {' '.join(cmd)}")

for s in INIT_SEEDS:
    print(f"===== init_seed={s} =====", flush=True)
    run([sys.executable, "scripts/train.py",
         "--workdir", DATA,
         "--out", f"/kaggle/working/run_m{M}_v2_s{s}",
         "--slice-num", str(M),
         "--epochs", str(EPOCHS),
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(s)])

In [ ]:
import glob, json

for path in sorted(glob.glob("/kaggle/working/run_m*/final_eval.json")):
    with open(path) as f:
        final = json.load(f)
    print(f"===== {path} (slice_num={final['args']['slice_num']}, "
          f"init_seed={final['args'].get('init_seed')}) =====")
    print(json.dumps(final["final"], indent=2))
    print("splits:", final["splits"])